# Velora Launcher — ZIP to APK
Upload the GitHub repository ZIP. This notebook installs the Android build toolchain and returns a debug APK.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Using:', zip_name)

In [ ]:
import os, pathlib, shutil, subprocess, zipfile
work = pathlib.Path('/content/velora-build')
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile('/content/' + zip_name) as z: z.extractall(work)
candidates = list(work.rglob('settings.gradle.kts')) + list(work.rglob('settings.gradle'))
if not candidates: raise RuntimeError('No Gradle Android project found in ZIP')
project = min((p.parent for p in candidates), key=lambda p: len(p.parts))
print('Project root:', project)

In [ ]:
subprocess.run('apt-get update -qq && apt-get install -y openjdk-17-jdk wget unzip > /dev/null', shell=True, check=True)
sdk = pathlib.Path('/content/android-sdk')
cmdline = sdk / 'cmdline-tools' / 'latest'
if not (cmdline / 'bin' / 'sdkmanager').exists():
    archive = pathlib.Path('/content/cmdline-tools.zip')
    subprocess.run(['wget','-q','https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip','-O',str(archive)], check=True)
    temp = pathlib.Path('/content/cmdline-temp')
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir()
    with zipfile.ZipFile(archive) as z: z.extractall(temp)
    cmdline.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(temp / 'cmdline-tools'), str(cmdline))
os.environ['ANDROID_SDK_ROOT'] = str(sdk)
os.environ['ANDROID_HOME'] = str(sdk)
sdkmanager = str(cmdline / 'bin' / 'sdkmanager')
subprocess.run(f'yes | {sdkmanager} --licenses > /dev/null', shell=True, check=False)
subprocess.run([sdkmanager, 'platform-tools', 'platforms;android-35', 'build-tools;35.0.0'], check=True)
(project / 'local.properties').write_text('sdk.dir=' + str(sdk) + '\n')
print('Android SDK ready')

In [ ]:
gradle_home = pathlib.Path('/content/gradle-8.9')
if not gradle_home.exists():
    gradle_zip = pathlib.Path('/content/gradle.zip')
    subprocess.run(['wget','-q','https://services.gradle.org/distributions/gradle-8.9-bin.zip','-O',str(gradle_zip)], check=True)
    with zipfile.ZipFile(gradle_zip) as z: z.extractall('/content')
gradle = str(gradle_home / 'bin' / 'gradle')
env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
subprocess.run([gradle, '--no-daemon', ':app:assembleDebug'], cwd=project, env=env, check=True)
apks = list((project / 'app' / 'build' / 'outputs' / 'apk' / 'debug').glob('*.apk'))
if not apks: raise RuntimeError('Build succeeded but APK was not found')
output = pathlib.Path('/content/Velora-debug.apk')
shutil.copy2(apks[0], output)
print('APK:', output, 'size:', round(output.stat().st_size/1024/1024, 2), 'MB')

In [ ]:
from google.colab import files
files.download('/content/Velora-debug.apk')